# 1. Producing the data  
In this task, we will implement Apache Kafka producers to simulate real-time data streaming. Spark and parallel data processing should not be used in this section, as we are simulating sensors that often lack processing capabilities.  

1. Every 1 second, load 50-100 accident records from the CSV file. You should keep a pointer in the file reading process and advance it per read. The data should be read in chronological order; just the number of records you read every second is a random number between 50 and 100 (inclusive).
2. Add the current timestamp (accident_ts) to the records. 
3. Send your batch of accident data to a Kafka topic with an appropriate name.

## Assignment 2B — Task 1: Kafka Producer
### Author: bgan0012
This notebook implements a pure Python Kafka producer that reads `streaming_collision.csv` chronologically and publishes batches of 50–100 rows per second to the Kafka topic `accident_stream`. Each message includes an `accident_ts` field set to the current Unix timestamp at send time.

### Imports

In [1]:
# ============================================================
# Task 1 — Imports
# Pure Python only — no Spark or PySpark imports permitted
# ============================================================

import csv          # parse streaming_collision.csv row by row
import json         # serialise each row as a JSON message
import time         # control the 1-second send interval
import random       # draw a fresh random batch size each second

from kafka import KafkaProducer  # pure Python Kafka client — kafka-python

print("All imports successful.")
import kafka
print(f"kafka-python version: {kafka.__version__}")

All imports successful.
kafka-python version: 2.2.3


## Configuration
Kafka broker address, topic name, and CSV path are defined as constants here. Kafka runs as a separate container named `kafka` on the `fit5202` Docker network — hence the broker address is `kafka:9092` rather than `localhost:9092`.

In [2]:
# ============================================================
# Task 1 — Producer Configuration
# ============================================================

# Kafka broker — runs as a separate container on the fit5202
# Docker network; address is kafka:9092, not localhost:9092
BOOTSTRAP_SERVERS = "kafka:9092"

# Topic name as specified in the assignment
TOPIC_NAME = "accident_stream"

# Path to the streaming collision CSV
CSV_PATH = "streaming_collision.csv"

# Batch size bounds (inclusive) per spec: random int in [50, 100]
BATCH_MIN = 50
BATCH_MAX = 100

print(f"Bootstrap server : {BOOTSTRAP_SERVERS}")
print(f"Topic            : {TOPIC_NAME}")
print(f"CSV path         : {CSV_PATH}")
print(f"Batch size range : {BATCH_MIN}–{BATCH_MAX} rows/second")

Bootstrap server : kafka:9092
Topic            : accident_stream
CSV path         : streaming_collision.csv
Batch size range : 50–100 rows/second


## Initialise Kafka Producer
The producer serialises each message value as a UTF-8 encoded JSON string. The key serialiser encodes `collision_index` as bytes to allow Kafka to route messages consistently by collision.

In [3]:
# ============================================================
# Task 1 — Initialise KafkaProducer
# ============================================================

# value_serializer: each message value is a dict — serialise to
# a JSON string then encode to UTF-8 bytes for Kafka transport.
# key_serializer: collision_index string → UTF-8 bytes.
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None
)

print(f"KafkaProducer initialised.")
print(f"Connected to : {BOOTSTRAP_SERVERS}")

KafkaProducer initialised.
Connected to : kafka:9092


## Kafka Producer Loop
The producer reads `streaming_collision.csv` using a persistent file pointer that advances per read, maintaining chronological order across the full run. Every second, a fresh random batch size between 50 and 100 rows is drawn. Each row is sent as a JSON message with all original fields serialised as strings, plus `accident_ts` added as a numeric Unix timestamp (integer).

In [ ]:
# ============================================================
# Task 1 — Producer Loop
#
# Design decisions:
# - csv.DictReader is re-opened each time the file is exhausted,
#   allowing the producer to loop indefinitely until manually
#   stopped. This ensures continuous data flow during the demo.
# - batch_size is re-drawn every second per spec.
# - accident_ts is added as a numeric Unix timestamp (int)
#   before sending — all other fields remain strings from
#   csv.DictReader and are sent as-is per spec.
# - producer.flush() after each batch ensures all buffered
#   messages are delivered before the 1-second sleep.
# - Loop runs until manually interrupted (KeyboardInterrupt).
# ============================================================

sent_total = 0   # running count of messages sent across all batches
batch_num  = 0   # batch counter for logging
loop_num   = 0   # how many times the CSV has been fully replayed

try:
    while True:
        loop_num += 1
        print(f"--- Starting CSV pass {loop_num} ---")

        with open(CSV_PATH, newline="", encoding="utf-8") as csv_file:
            reader = csv.DictReader(csv_file)  # fresh pointer each pass

            while True:
                # Draw a fresh random batch size each second per spec
                batch_size = random.randint(BATCH_MIN, BATCH_MAX)
                batch      = []

                # Read the next batch_size rows from current pointer
                for _ in range(batch_size):
                    row = next(reader, None)  # None = end of file
                    if row is None:
                        break
                    batch.append(row)

                # Inner loop exhausted — break to outer loop to reopen
                if not batch:
                    print(f"CSV pass {loop_num} complete. "
                          f"Total sent so far: {sent_total:,}")
                    break

                # Add accident_ts as numeric Unix timestamp (integer)
                # All other fields remain as strings from DictReader
                current_ts = int(time.time())
                for row in batch:
                    row["accident_ts"] = current_ts

                # Send each row to Kafka as a JSON message
                for row in batch:
                    producer.send(
                        TOPIC_NAME,
                        key=row.get("collision_index", ""),
                        value=row
                    )

                # Flush ensures delivery before sleeping
                producer.flush()

                batch_num  += 1
                sent_total += len(batch)

                print(f"Batch {batch_num:>4} | "
                      f"rows: {len(batch):>3} | "
                      f"total: {sent_total:>7,} | "
                      f"pass: {loop_num} | "
                      f"ts: {current_ts}")

                time.sleep(1)  # wait 1 second before next batch

except KeyboardInterrupt:
    print(f"\nProducer stopped manually.")
    print(f"Total batches sent : {batch_num:,}")
    print(f"Total messages sent: {sent_total:,}")
    print(f"CSV passes completed: {loop_num}")

--- Starting CSV pass 1 ---
Batch    1 | rows:  91 | total:      91 | pass: 1 | ts: 1781502465
Batch    2 | rows:  62 | total:     153 | pass: 1 | ts: 1781502466
Batch    3 | rows:  69 | total:     222 | pass: 1 | ts: 1781502467
Batch    4 | rows:  77 | total:     299 | pass: 1 | ts: 1781502468
Batch    5 | rows:  51 | total:     350 | pass: 1 | ts: 1781502469
Batch    6 | rows:  90 | total:     440 | pass: 1 | ts: 1781502470
Batch    7 | rows:  55 | total:     495 | pass: 1 | ts: 1781502471
Batch    8 | rows:  51 | total:     546 | pass: 1 | ts: 1781502472
Batch    9 | rows:  55 | total:     601 | pass: 1 | ts: 1781502473
Batch   10 | rows:  55 | total:     656 | pass: 1 | ts: 1781502474
Batch   11 | rows:  70 | total:     726 | pass: 1 | ts: 1781502475
Batch   12 | rows:  52 | total:     778 | pass: 1 | ts: 1781502476
Batch   13 | rows:  89 | total:     867 | pass: 1 | ts: 1781502477
Batch   14 | rows:  88 | total:     955 | pass: 1 | ts: 1781502478
Batch   15 | rows:  99 | total:   

Batch  123 | rows:  60 | total:   9,109 | pass: 1 | ts: 1781502588
Batch  124 | rows:  94 | total:   9,203 | pass: 1 | ts: 1781502589
Batch  125 | rows:  89 | total:   9,292 | pass: 1 | ts: 1781502590
Batch  126 | rows:  51 | total:   9,343 | pass: 1 | ts: 1781502591
Batch  127 | rows:  86 | total:   9,429 | pass: 1 | ts: 1781502592
Batch  128 | rows:  63 | total:   9,492 | pass: 1 | ts: 1781502593
Batch  129 | rows:  68 | total:   9,560 | pass: 1 | ts: 1781502594
Batch  130 | rows:  92 | total:   9,652 | pass: 1 | ts: 1781502595
Batch  131 | rows:  66 | total:   9,718 | pass: 1 | ts: 1781502596
Batch  132 | rows:  72 | total:   9,790 | pass: 1 | ts: 1781502597
Batch  133 | rows:  93 | total:   9,883 | pass: 1 | ts: 1781502598
Batch  134 | rows:  52 | total:   9,935 | pass: 1 | ts: 1781502599
Batch  135 | rows:  90 | total:  10,025 | pass: 1 | ts: 1781502600
Batch  136 | rows:  76 | total:  10,101 | pass: 1 | ts: 1781502601
Batch  137 | rows:  62 | total:  10,163 | pass: 1 | ts: 178150

Batch  246 | rows:  74 | total:  18,339 | pass: 1 | ts: 1781502713
Batch  247 | rows:  51 | total:  18,390 | pass: 1 | ts: 1781502714
Batch  248 | rows:  57 | total:  18,447 | pass: 1 | ts: 1781502715
Batch  249 | rows:  74 | total:  18,521 | pass: 1 | ts: 1781502716
Batch  250 | rows:  81 | total:  18,602 | pass: 1 | ts: 1781502717
Batch  251 | rows:  85 | total:  18,687 | pass: 1 | ts: 1781502718
Batch  252 | rows:  72 | total:  18,759 | pass: 1 | ts: 1781502719
Batch  253 | rows:  82 | total:  18,841 | pass: 1 | ts: 1781502720
Batch  254 | rows:  57 | total:  18,898 | pass: 1 | ts: 1781502721
Batch  255 | rows:  80 | total:  18,978 | pass: 1 | ts: 1781502722
Batch  256 | rows:  83 | total:  19,061 | pass: 1 | ts: 1781502723
Batch  257 | rows:  99 | total:  19,160 | pass: 1 | ts: 1781502724
Batch  258 | rows:  71 | total:  19,231 | pass: 1 | ts: 1781502725
Batch  259 | rows:  74 | total:  19,305 | pass: 1 | ts: 1781502726
Batch  260 | rows:  98 | total:  19,403 | pass: 1 | ts: 178150

Batch  369 | rows:  62 | total:  27,455 | pass: 1 | ts: 1781502838
Batch  370 | rows:  73 | total:  27,528 | pass: 1 | ts: 1781502839
Batch  371 | rows:  55 | total:  27,583 | pass: 1 | ts: 1781502840
Batch  372 | rows:  54 | total:  27,637 | pass: 1 | ts: 1781502841
Batch  373 | rows:  59 | total:  27,696 | pass: 1 | ts: 1781502842
Batch  374 | rows:  54 | total:  27,750 | pass: 1 | ts: 1781502843
Batch  375 | rows:  58 | total:  27,808 | pass: 1 | ts: 1781502845
Batch  376 | rows:  69 | total:  27,877 | pass: 1 | ts: 1781502846
Batch  377 | rows:  67 | total:  27,944 | pass: 1 | ts: 1781502847
Batch  378 | rows:  60 | total:  28,004 | pass: 1 | ts: 1781502848
Batch  379 | rows:  50 | total:  28,054 | pass: 1 | ts: 1781502849
Batch  380 | rows:  92 | total:  28,146 | pass: 1 | ts: 1781502850
Batch  381 | rows:  86 | total:  28,232 | pass: 1 | ts: 1781502851
Batch  382 | rows:  50 | total:  28,282 | pass: 1 | ts: 1781502852
Batch  383 | rows:  79 | total:  28,361 | pass: 1 | ts: 178150

Batch  491 | rows:  96 | total:  36,440 | pass: 2 | ts: 1781502963
Batch  492 | rows:  71 | total:  36,511 | pass: 2 | ts: 1781502964
Batch  493 | rows:  74 | total:  36,585 | pass: 2 | ts: 1781502965
Batch  494 | rows:  51 | total:  36,636 | pass: 2 | ts: 1781502966
Batch  495 | rows:  86 | total:  36,722 | pass: 2 | ts: 1781502967
Batch  496 | rows:  72 | total:  36,794 | pass: 2 | ts: 1781502969
Batch  497 | rows:  51 | total:  36,845 | pass: 2 | ts: 1781502970
Batch  498 | rows:  74 | total:  36,919 | pass: 2 | ts: 1781502971
Batch  499 | rows:  93 | total:  37,012 | pass: 2 | ts: 1781502972
Batch  500 | rows:  51 | total:  37,063 | pass: 2 | ts: 1781502973
Batch  501 | rows:  60 | total:  37,123 | pass: 2 | ts: 1781502974
Batch  502 | rows:  57 | total:  37,180 | pass: 2 | ts: 1781502975
Batch  503 | rows:  56 | total:  37,236 | pass: 2 | ts: 1781502976
Batch  504 | rows:  73 | total:  37,309 | pass: 2 | ts: 1781502977
Batch  505 | rows:  64 | total:  37,373 | pass: 2 | ts: 178150

...

Batch 1228 | rows:  78 | total:  91,164 | pass: 3 | ts: 1781503718
Batch 1229 | rows:  82 | total:  91,246 | pass: 3 | ts: 1781503719
Batch 1230 | rows:  58 | total:  91,304 | pass: 3 | ts: 1781503720
Batch 1231 | rows:  68 | total:  91,372 | pass: 3 | ts: 1781503721
Batch 1232 | rows:  89 | total:  91,461 | pass: 3 | ts: 1781503722
Batch 1233 | rows:  84 | total:  91,545 | pass: 3 | ts: 1781503723
Batch 1234 | rows:  84 | total:  91,629 | pass: 3 | ts: 1781503724
Batch 1235 | rows:  87 | total:  91,716 | pass: 3 | ts: 1781503725
Batch 1236 | rows:  61 | total:  91,777 | pass: 3 | ts: 1781503726
Batch 1237 | rows:  82 | total:  91,859 | pass: 3 | ts: 1781503727
Batch 1238 | rows:  74 | total:  91,933 | pass: 3 | ts: 1781503728
Batch 1239 | rows:  55 | total:  91,988 | pass: 3 | ts: 1781503729
Batch 1240 | rows:  90 | total:  92,078 | pass: 3 | ts: 1781503730
Batch 1241 | rows:  92 | total:  92,170 | pass: 3 | ts: 1781503731
Batch 1242 | rows:  52 | total:  92,222 | pass: 3 | ts: 178150

Batch 1350 | rows:  72 | total: 100,288 | pass: 4 | ts: 1781503841
Batch 1351 | rows:  67 | total: 100,355 | pass: 4 | ts: 1781503843
Batch 1352 | rows:  52 | total: 100,407 | pass: 4 | ts: 1781503844
Batch 1353 | rows:  58 | total: 100,465 | pass: 4 | ts: 1781503845
Batch 1354 | rows:  72 | total: 100,537 | pass: 4 | ts: 1781503846
Batch 1355 | rows:  85 | total: 100,622 | pass: 4 | ts: 1781503847
Batch 1356 | rows:  88 | total: 100,710 | pass: 4 | ts: 1781503848
Batch 1357 | rows:  80 | total: 100,790 | pass: 4 | ts: 1781503849
Batch 1358 | rows:  65 | total: 100,855 | pass: 4 | ts: 1781503850
Batch 1359 | rows:  75 | total: 100,930 | pass: 4 | ts: 1781503851
Batch 1360 | rows:  72 | total: 101,002 | pass: 4 | ts: 1781503852
Batch 1361 | rows:  78 | total: 101,080 | pass: 4 | ts: 1781503853
Batch 1362 | rows:  80 | total: 101,160 | pass: 4 | ts: 1781503854
Batch 1363 | rows:  76 | total: 101,236 | pass: 4 | ts: 1781503855
Batch 1364 | rows:  86 | total: 101,322 | pass: 4 | ts: 178150

Batch 1473 | rows:  65 | total: 109,583 | pass: 4 | ts: 1781503966
Batch 1474 | rows:  81 | total: 109,664 | pass: 4 | ts: 1781503967
Batch 1475 | rows:  82 | total: 109,746 | pass: 4 | ts: 1781503968
Batch 1476 | rows:  75 | total: 109,821 | pass: 4 | ts: 1781503969
Batch 1477 | rows:  92 | total: 109,913 | pass: 4 | ts: 1781503970
Batch 1478 | rows:  81 | total: 109,994 | pass: 4 | ts: 1781503971
Batch 1479 | rows:  72 | total: 110,066 | pass: 4 | ts: 1781503972
Batch 1480 | rows:  95 | total: 110,161 | pass: 4 | ts: 1781503973
Batch 1481 | rows:  97 | total: 110,258 | pass: 4 | ts: 1781503974
Batch 1482 | rows:  93 | total: 110,351 | pass: 4 | ts: 1781503975
Batch 1483 | rows:  73 | total: 110,424 | pass: 4 | ts: 1781503976
Batch 1484 | rows:  97 | total: 110,521 | pass: 4 | ts: 1781503977
Batch 1485 | rows:  73 | total: 110,594 | pass: 4 | ts: 1781503978
Batch 1486 | rows: 100 | total: 110,694 | pass: 4 | ts: 1781503980
Batch 1487 | rows:  61 | total: 110,755 | pass: 4 | ts: 178150

Batch 1596 | rows:  99 | total: 119,320 | pass: 4 | ts: 1781504092
Batch 1597 | rows:  55 | total: 119,375 | pass: 4 | ts: 1781504093
Batch 1598 | rows:  70 | total: 119,445 | pass: 4 | ts: 1781504094
Batch 1599 | rows:  78 | total: 119,523 | pass: 4 | ts: 1781504095
Batch 1600 | rows:  68 | total: 119,591 | pass: 4 | ts: 1781504096
Batch 1601 | rows: 100 | total: 119,691 | pass: 4 | ts: 1781504097
Batch 1602 | rows: 100 | total: 119,791 | pass: 4 | ts: 1781504098
Batch 1603 | rows:  80 | total: 119,871 | pass: 4 | ts: 1781504099
Batch 1604 | rows:  70 | total: 119,941 | pass: 4 | ts: 1781504100
Batch 1605 | rows:  79 | total: 120,020 | pass: 4 | ts: 1781504101
Batch 1606 | rows:  93 | total: 120,113 | pass: 4 | ts: 1781504102
Batch 1607 | rows:  78 | total: 120,191 | pass: 4 | ts: 1781504103
Batch 1608 | rows:  82 | total: 120,273 | pass: 4 | ts: 1781504104
Batch 1609 | rows:  79 | total: 120,352 | pass: 4 | ts: 1781504105
Batch 1610 | rows: 100 | total: 120,452 | pass: 4 | ts: 178150

Batch 1719 | rows:  91 | total: 128,676 | pass: 4 | ts: 1781504217
Batch 1720 | rows:  59 | total: 128,735 | pass: 4 | ts: 1781504218
Batch 1721 | rows:  57 | total: 128,792 | pass: 4 | ts: 1781504219
Batch 1722 | rows:  87 | total: 128,879 | pass: 4 | ts: 1781504220
Batch 1723 | rows: 100 | total: 128,979 | pass: 4 | ts: 1781504221
Batch 1724 | rows:  54 | total: 129,033 | pass: 4 | ts: 1781504222
Batch 1725 | rows:  70 | total: 129,103 | pass: 4 | ts: 1781504223
Batch 1726 | rows: 100 | total: 129,203 | pass: 4 | ts: 1781504224
Batch 1727 | rows:  87 | total: 129,290 | pass: 4 | ts: 1781504225
Batch 1728 | rows:  57 | total: 129,347 | pass: 4 | ts: 1781504226
Batch 1729 | rows:  63 | total: 129,410 | pass: 4 | ts: 1781504227
Batch 1730 | rows:  54 | total: 129,464 | pass: 4 | ts: 1781504228
Batch 1731 | rows:  88 | total: 129,552 | pass: 4 | ts: 1781504229
Batch 1732 | rows:  78 | total: 129,630 | pass: 4 | ts: 1781504230
Batch 1733 | rows:  76 | total: 129,706 | pass: 4 | ts: 178150

## References

### Generative AI Usage

The following table documents all use of generative AI tools in accordance with the Monash University Generative AI Assessment Policy (FIT5202, Semester 1 2026).

| Tool | Version | Cell | Purpose | Representative Prompt |
|------|---------|------|---------|----------------------|
| Claude | Sonnet 4.6 (claude.ai) | Imports | Confirming pure Python constraints and library selection for Task 1 | "Task 1 must use pure Python only — no Spark or PySpark imports. What libraries are needed to build a Kafka producer that reads a CSV and sends JSON messages?" |
| Claude | Sonnet 4.6 (claude.ai) | Configuration | Clarifying Kafka broker address inside Docker network | "Why should the bootstrap server be kafka:9092 and not localhost:9092 when running inside the fit5202 Docker container?" |
| Claude | Sonnet 4.6 (claude.ai) | Initialise Kafka Producer | Serialiser design for mixed-type message values | "How should I configure the KafkaProducer value_serializer to send a dict where most fields are strings but accident_ts is an integer?" |
| Claude | Sonnet 4.6 (claude.ai) | Producer Loop | Persistent file pointer design and loop-back logic for continuous streaming | "How do I keep a persistent file pointer that advances per read in Python, and loop back to the start of the CSV when it is exhausted, while maintaining chronological order?" |